# Run Presidio
Stanza NER 

## Set Up

### Import packages

In [ ]:
# Install packages if not available on your cluster
import pandas as pd
import numpy as np
from presidio_analyzer import AnalyzerEngine, BatchAnalyzerEngine
from presidio_analyzer.nlp_engine import StanzaNlpEngine, NerModelConfiguration
import stanza

In [ ]:
stanza.download("en")  # download English model

### Spark Setup

In [ ]:
# Configure Spark session
spark.conf.set("spark.sql.shuffle.partitions", "400")
spark.conf.set("spark.sql.execution.arrow.enabled", "true")
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "true")

sc = spark.sparkContext

In [ ]:
# Review configuration details
for item in spark.sparkContext.getConf().getAll():
    print(item)

### Set parameters

See available Presidio entity types: https://microsoft.github.io/presidio/supported_entities/

In [ ]:
data_path = "catalog.schema.table"
data = spark.sql("""select * from """ + data_path)
target_col = "MY_TARGET"
id_cols_str = "COL1, COL2"
id_col_list = id_cols_str.split(",")
id_col_list = [c.strip() for c in id_col_list]
output_table_name = "catalog.schema.output_table"
access_group = "group-name"

In [ ]:
my_entities_list = [
    "PERSON",
    "NRP",
    "DATE_TIME",
    "LOCATION",
    "EMAIL_ADDRESS",
    "PHONE_NUMBER",
    "US_SSN",
    "ORGANIZATION",
]  # Adjust as needed
my_allow_list = []  # Terms for the model NOT to mark as PII
my_deny_list = []  # Terms for the model to ALWAYS mark as PII

## Set up Stanza

In [ ]:
# Define which model to use
model_config = [{"lang_code": "en", "model_name": "en"}]

# Define which entities the model returns and how they map to Presidio's
entity_mapping = {
    "PERSON": "PERSON",
    "NORP": "NRP",
    "FACILITY": "LOCATION",
    "FAC": "LOCATION",
    "ORGANIZATION": "ORGANIZATION",
    "ORG": "LOCATION",
    "GPE": "LOCATION",
    "LOCATION": "LOCATION",
    "DATE": "DATE_TIME",
    "TIME": "DATE_TIME",
}

my_labels_to_ignore = [
    "PRODUCT",
    "EVENT",
    "WORK_OF_ART",
    "LAW",
    "PERCENT",
    "MONEY",
    "QUANTITY",
    "ORDINAL",
    "CARDINAL",
]

ner_model_configuration = NerModelConfiguration(
    model_to_presidio_entity_mapping=entity_mapping,
    labels_to_ignore=my_labels_to_ignore,
)

# Create the Stanza NLP Engine based on this configuration
stanza_nlp_engine = StanzaNlpEngine(
    models=model_config, ner_model_configuration=ner_model_configuration
)

## Run Presidio Analyzer
Used batch processing instead of broadcasting as broadcasting did not work with Stanza architecture

In [ ]:
# Prepare data for batch processing required format
df = data
df_dict = df.toPandas().to_dict(orient="list")

keys_to_keep = id_col_list + [target_col]
df_dict = {k: df_dict[k] for k in keys_to_keep if k in df_dict}

In [ ]:
# Initialize Presidio's AnalyzerEngine and Batch Analyzer
analyzer = AnalyzerEngine(nlp_engine=stanza_nlp_engine)
batch_analyzer = BatchAnalyzerEngine(analyzer_engine=analyzer)

In [ ]:
# Define columns that do not need to be checked

my_keys_to_skip = []
for col_name in df.columns:
    if col_name != target_col:
        my_keys_to_skip.append(col_name)

In [ ]:
analyzer_results = batch_analyzer.analyze_dict(
    df_dict,
    language="en",
    entities=my_entities_list,
    allow_list=my_allow_list,
    keys_to_skip=my_keys_to_skip,
)
analyzer_results = list(analyzer_results)
analyzer_results_raw_df = pd.DataFrame(analyzer_results)

### Format results

In [ ]:
results_list = analyzer_results_raw_df.loc[
    analyzer_results_raw_df["key"] == target_col, "recognizer_results"
].tolist()[0]
results_data = {"raw_results": results_list}

In [ ]:
df_dict.update(results_data)

In [ ]:
raw_results_df = pd.DataFrame(df_dict)
raw_results_df = raw_results_df.astype(str)
raw_results_df["detected_entities"] = np.where(
    raw_results_df["raw_results"] != "[]", raw_results_df["raw_results"], ""
)

In [ ]:
# Convert Pandas DataFrame to PySpark DataFrame
spark_df = spark.createDataFrame(raw_results_df)

## Preview Results

In [ ]:
spark_df.show()

In [ ]:
spark.sql("select count(*) from {df}", df=spark_df).show()

## Save Results

In [ ]:
spark_df.write.mode("overwrite").saveAsTable(output_table_name)

In [ ]:
spark.sql("GRANT SELECT ON " + output_table_name + " TO " + access_group)